# JAM Knowledge Mutation 001

Formal hosted-GPU orchestration for the frozen release-oriented JAM mutation protocol. Do not merge the experiment PR until all intended formal result commits have been published.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

ROOT = Path('/kaggle/working/mini-cells')
BRANCH = 'codex/jam-knowledge-mutation-001'
if not ROOT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/ArcheLabs/mini-cells.git', str(ROOT)], check=True)
subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=ROOT, check=True)
subprocess.run(['git', 'checkout', '-B', BRANCH, f'origin/{BRANCH}'], cwd=ROOT, check=True)
print({'head': subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=ROOT, text=True).strip()})

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==5.0.0', 'huggingface_hub==1.11.0', 'safetensors==0.7.0'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[lm,dev]'], cwd=ROOT, check=True)
subprocess.run([sys.executable, 'scripts/research/jam_knowledge_v0_1/validate_dataset.py'], cwd=ROOT, check=True)
subprocess.run([sys.executable, 'scripts/research/jam_knowledge_mutation_001/dataset_identity.py'], cwd=ROOT, check=True)

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ['GITHUB_TOKEN'] = secrets.get_secret('GITHUB_TOKEN')
    try:
        os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
    except Exception:
        pass
except Exception as exc:
    raise RuntimeError('Kaggle Secret GITHUB_TOKEN is required') from exc

subprocess.run([
    sys.executable, 'scripts/research/jam_knowledge_mutation_001/publish.py',
    '--branch', BRANCH, '--preflight-only'
], cwd=ROOT, check=True)

In [ ]:
import torch, transformers, huggingface_hub, safetensors
assert torch.cuda.is_available(), 'CUDA is required for the formal run'
protocol = json.loads((ROOT / 'research/validations/jam-knowledge-mutation-001/protocol.json').read_text())
formal_seeds = protocol['formal_seeds']
free_mb, _total_mb = torch.cuda.mem_get_info(0)
free_mb //= 1024 * 1024
assert free_mb >= 14000, f'need >= 14000 MiB free GPU memory, found {free_mb}'
print({
    'gpu': torch.cuda.get_device_name(0),
    'free_mb': free_mb,
    'torch': torch.__version__,
    'transformers': transformers.__version__,
    'huggingface_hub': huggingface_hub.__version__,
    'safetensors': safetensors.__version__,
    'protocol_version': protocol['protocol_version'],
    'formal_seeds': formal_seeds,
    'capacity_ladder': protocol['mutation']['capacity_ladder'],
    'dataset': protocol['dataset']['id'],
    'dataset_manifest_sha256': protocol['dataset']['manifest_sha256'],
})

In [ ]:
def run_compact(command, log_path):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('w', encoding='utf-8') as handle:
        process = subprocess.Popen(
            command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, env={**os.environ, 'HF_HUB_DISABLE_PROGRESS_BARS': '1', 'TRANSFORMERS_NO_ADVISORY_WARNINGS': '1'}
        )
        assert process.stdout is not None
        for line in process.stdout:
            handle.write(line)
            handle.flush()
            if line.startswith('[jam001]'):
                print(line, end='')
        returncode = process.wait()
    if returncode != 0:
        tail = log_path.read_text(encoding='utf-8', errors='replace').splitlines()[-80:]
        print('\n=== Child log tail ===')
        print('\n'.join(tail))
        subprocess.run(['nvidia-smi'], check=False)
        raise RuntimeError(f'formal child failed with exit code {returncode}; full log: {log_path}')

artifact_root = ROOT / 'artifacts/experiments/jam-knowledge-mutation-001'
for seed in formal_seeds:
    durable = artifact_root / f'seed-{seed}/seed_summary.json'
    if durable.is_file():
        print(f'[jam001][seed={seed}] already published; skipping')
        continue
    free_mb, _ = torch.cuda.mem_get_info(0)
    free_mb //= 1024 * 1024
    assert free_mb >= 14000, f'need >= 14000 MiB free before seed {seed}, found {free_mb}'
    run_compact([
        sys.executable, 'scripts/research/jam_knowledge_mutation_001/run_formal_seed.py',
        '--seed', str(seed), '--device', 'cuda:0'
    ], ROOT / f'results/jam-knowledge-mutation-001-launcher/seed-{seed}.log')
    subprocess.run([
        sys.executable, 'scripts/research/jam_knowledge_mutation_001/publish.py',
        '--seed', str(seed), '--branch', BRANCH
    ], cwd=ROOT, check=True)
    decision = json.loads((artifact_root / 'decision.json').read_text())
    print({
        'status': decision['status'],
        'completed_seeds': decision['completed_seeds'],
        'passed_seeds': decision['passed_seeds'],
        'selected_capacity_by_seed': decision['selected_capacity_by_seed'],
    })

In [ ]:
decision_path = ROOT / 'artifacts/experiments/jam-knowledge-mutation-001/decision.json'
if decision_path.is_file():
    decision = json.loads(decision_path.read_text())
    print(json.dumps({
        'status': decision['status'],
        'scientific_decision': decision['scientific_decision'],
        'completed_seeds': decision['completed_seeds'],
        'passed_seeds': decision['passed_seeds'],
        'minimum_passing_capacity_observed': decision['minimum_passing_capacity_observed'],
    }, indent=2))

Recovery rule: rerun the notebook. Every completed formal seed, including scientific FAIL, is committed and pushed immediately. Full child logs remain under `results/jam-knowledge-mutation-001-launcher/` and are not streamed wholesale into the notebook page.